<a href="https://colab.research.google.com/github/itsrealfarman/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/itsrealfarman/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook summarizes the full track (ML-01 through ML-10) into the same seven sections as the deployed paper. Full detail and every executed step live in the individual weekly notebooks — this is the connective summary.

**Deployed paper:** see `submission/paper_url.txt` for the live URL.


In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

import os, getpass, json as jsonlib

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb, pandas as pd, numpy as np

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
MONTH_START, MID_MONTH, MONTH_END = "2026-03-01", "2026-03-16", "2026-04-01"
print("Connected.")


Connected.


## 1. Question

*The research question and the decision it supports.*

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring.

**Question:** Which pages should a content reviewer open first, this week, given limited review capacity?

**Decision supported:** which pages a reviewer opens first out of thousands. **Action taken by a human:** refresh, rewrite title/meta, monitor, or leave alone. **Cost of a wrong call:** a false positive wastes limited reviewer time; a false negative lets a genuinely declining page keep losing visibility unnoticed. This is a ranking/scoring problem, not a plain classification problem — the deliverable is an ordered queue with reasons, not a single label.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** `flyrank_pseudonymized_warehouse_release_v20260703` (Hugging Face, gated, read-only via DuckDB).
**Tables:** `dim_clients`, `fact_content_daily_performance`.
**Window:** March 2026 (mid-panel month) — the final warehouse month was deliberately left as a sealed test window, never touched.
**Feature/label split:** Mar 1–15 (features) vs Mar 16–31 (label) within the same month.
**Volume floor:** ≥500 impressions in the feature window.
**Excluded, on purpose:** `fact_content_query_90d` (query-level table) — useful for future work, left out to keep the first data contract small enough to verify carefully.
**Limitation:** the panel is unbalanced across clients — `gsc_data_start` differs per client.


In [3]:
clients = con.sql(f"SELECT COUNT(*) AS n_clients FROM {TABLES['dim_clients']}").df()
print(clients)


   n_clients
0        104


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label:** `is_declining = imp_h2 < 0.8 × avg_impressions_h1 × 15` — a same-month, percentage-threshold proxy, not a verified business outcome.

**Features (all first-half-only):** `avg_impressions_h1`, `avg_clicks_h1`, `ctr_h1`, `avg_position_h1`, `days_with_impressions_h1`.

**Baseline:** hand-written rule `ctr_below_tier_expectation` — flags top_10/top_20-tier pages with CTR meaningfully below their tier's own average. Its two supporting signals were checked against real bucketed data first — both CONFIRMED.

**Model:** client-grouped Random Forest (300 trees, max depth 8, balanced class weight).

**Validation:** 75/25 split grouped by `client_hash_id` (`GroupShuffleSplit`), zero client overlap confirmed.

**Leakage checks (two found and fixed):**
1. Feature-level — including `imp_h2` (the label's own source) spiked AUC to 0.999. Even without it, a low volume floor (50) let percentage-noise leak through `avg_impressions_h1` alone — raising the floor to 500 fixed it (honest AUC 0.614).
2. Split-level — a naive random split scored 0.82 Precision@50 vs the honest client-grouped split's 0.56 — the gap is the optimism client leakage bought for free.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

| Method | Precision@50 |
|---|---|
| Baseline rule (`ctr_below_tier_expectation`) | 0.28 |
| Model, client-grouped split (ML-06 run) | 0.66 |
| Model, client-grouped split (ML-09 re-run) | 0.56 |

Reported as a range (0.56–0.66), not a single number — the exact figure moved between two honest re-runs of the identical method, given a modest number of held-out clients. Both are several times the baseline's 0.28.

**Top driver:** `ctr_h1` (built-in importance 0.331, permutation importance 0.057 — both far ahead of any other feature). **Known failure mode:** false positives cluster on already-bottomed-out pages (very poor position + near-zero CTR) rather than genuinely worsening ones — handled in the playbook with a dedicated `monitor_only` action.


In [5]:
results_summary = {
    "baseline_precision_at_50": 0.28,
    "model_precision_at_50_run1_ML06": 0.66,
    "model_precision_at_50_run2_ML09": 0.56,
    "top_feature": "ctr_h1",
}
print(jsonlib.dumps(results_summary, indent=2))


{
  "baseline_precision_at_50": 0.28,
  "model_precision_at_50_run1_ML06": 0.66,
  "model_precision_at_50_run2_ML09": 0.56,
  "top_feature": "ctr_h1"
}


## 5. Limitations

*What this work cannot claim.*

- **Scope:** one mid-panel month (March 2026), one 104-client slice — not yet tested on the sealed final month or the full multi-year warehouse.
- **Label:** a same-month, percentage-threshold proxy, not a verified business decline.
- **Causality:** nothing here shows that acting on a flagged page causes recovery — this mirrors a gap found while auditing FlyRank's own published research paper (its "52× impression lift" refresh finding was not a randomized comparison either).
- **Variance:** Precision@50 moved from 0.66 to 0.56 across two honest re-runs — report a range, not a fixed number.
- **Generalization:** the reported range is an average across held-out clients; any single new client could sit well above or below it.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Final queue: **41,816 pages** — `no_action` 31,209 · `review_title_meta` 5,400 · `refresh_content` 3,246 · `monitor_only` 1,961.

**Never automate:** auto-publishing rewrites without human sign-off; auto-merging/redirecting pages; treating a high decline probability as proof a fix will work; scoring clients absent from training data without re-validating.

**Retrain trigger:** monthly re-check on a fresh client-grouped holdout — if Precision@50 drifts toward the baseline's 0.28, retrain.


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Re-generating the same three artifacts committed in ML-10: the ranked queue, the metrics JSON, and the action-count figure — these are the receipts behind the paper's Results and Recommendations sections.


In [8]:
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

capstone_metrics = {
    "baseline_precision_at_50": 0.28,
    "model_precision_at_50_range": [0.56, 0.66],
    "leak_checks": {
        "feature_level": {"leaky_auc": 0.999, "honest_auc_after_fix": 0.614},
        "split_level": {"naive_split_p50": 0.82, "grouped_split_p50": 0.56},
    },
    "queue_size": 41816,
    "action_counts": {
        "no_action": 31209,
        "review_title_meta": 5400,
        "refresh_content": 3246,
        "monitor_only": 1961,
    },
    "paper_url_file": "submission/paper_url.txt",
}

with open("work/outputs/capstone_summary.json", "w") as f:
    jsonlib.dump(capstone_metrics, f, indent=2)

print("Wrote work/outputs/capstone_summary.json")
print(jsonlib.dumps(capstone_metrics, indent=2))


Wrote work/outputs/capstone_summary.json
{
  "baseline_precision_at_50": 0.28,
  "model_precision_at_50_range": [
    0.56,
    0.66
  ],
  "leak_checks": {
    "feature_level": {
      "leaky_auc": 0.999,
      "honest_auc_after_fix": 0.614
    },
    "split_level": {
      "naive_split_p50": 0.82,
      "grouped_split_p50": 0.56
    }
  },
  "queue_size": 41816,
  "action_counts": {
    "no_action": 31209,
    "review_title_meta": 5400,
    "refresh_content": 3246,
    "monitor_only": 1961
  },
  "paper_url_file": "submission/paper_url.txt"
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] All seven sections filled, matching the deployed paper
- [ ] Results reported as an honest range, not a single inflated number
- [ ] Limitations and no-go list both present
- [ ] `submission/paper_url.txt` contains the exact live URL, one line, nothing else
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] Paper checked for both required sections: Abstract (top) and Acknowledgments + flyrank.ai credit (bottom)
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
